# 🎬 Pre-Release Movie Success Prediction
## Phases 2–4: Feature Engineering → Model Training → Evaluation
### Now combining TMDB metadata + Reddit sentiment + YouTube trailer sentiment/engagement

| | |
|---|---|
| **Data Sources** | TMDB (metadata) · Reddit (post sentiment) · YouTube (trailer views/likes + comment sentiment) |
| **Targets** | Box Office Revenue (USD) · Audience Rating (1–10 scale) |
| **Models** | Random Forest · Gradient Boosting (sklearn `GradientBoostingRegressor`, used in place of XGBoost — swap back to `xgb.XGBRegressor` if you have `xgboost` installed) |
| **Evaluation** | MAE · RMSE · R² |
| **Output** | Continuous predictions → Blockbuster Hit / Commercial Success / Critical Darling / Flop |

## ⚙️ 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, json
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.preprocessing   import MultiLabelBinarizer
from sklearn.ensemble        import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics         import mean_absolute_error, mean_squared_error, r2_score
from sklearn.impute          import SimpleImputer

# NOTE: xgboost was not available in the environment that generated this notebook.
# If you have xgboost installed, you can swap GradientBoostingRegressor below
# for xgb.XGBRegressor (just `import xgboost as xgb` and use xgb.XGBRegressor(...)).

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print("✅ Libraries loaded")

## 📂 2. Load Data

In [ ]:
tmdb    = pd.read_csv('tmdb_movies_cleaned.csv')
reddit  = pd.read_csv('reddit_sentiment_results.csv')
youtube = pd.read_csv('youtube_sentiment_results.csv')

print(f"TMDB    : {tmdb.shape}")
print(f"Reddit  : {reddit.shape}")
print(f"YouTube : {youtube.shape}")
print()
tmdb.head(3)

---
## 🛠️ Phase 2 — Feature Engineering

### 2.1 Merge TMDB + Reddit + YouTube

In [ ]:
REDDIT_COLS = ['Title','post_count','avg_compound','avg_positive','avg_negative',
               'avg_neutral','positive_ratio','negative_ratio','total_score','avg_num_comments']

# YouTube: rename sentiment cols to avoid clashing with Reddit's identical names,
# and drop like_ratio (constant = 1.0 across all 197 rows, no signal — likely a
# YouTube API quirk since public dislike counts were removed).
YOUTUBE_RENAME = {
    'avg_compound':   'yt_avg_compound',
    'avg_positive':   'yt_avg_positive',
    'avg_negative':   'yt_avg_negative',
    'avg_neutral':    'yt_avg_neutral',
    'positive_ratio': 'yt_positive_ratio',
    'negative_ratio': 'yt_negative_ratio',
}
youtube = youtube.rename(columns=YOUTUBE_RENAME)
YOUTUBE_COLS = ['Title','video_view_count','likes','trailer_score','comment_count',
                'yt_avg_compound','yt_avg_positive','yt_avg_negative','yt_avg_neutral',
                'yt_positive_ratio','yt_negative_ratio','total_likes','weighted_compound']

df = tmdb.merge(reddit[REDDIT_COLS], on='Title', how='left')
df = df.merge(youtube[YOUTUBE_COLS], on='Title', how='left')
print(f"Merged: {df.shape}")

# Convert numeric columns
for col in ['Budget','Revenue','Rating','Vote_Count','Runtime']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df['Year'] = df['Year'].astype(str).str.split('.').str[0].astype(int)

# Reddit numeric conversion
for col in ['post_count','avg_compound','avg_positive','avg_negative',
            'avg_neutral','positive_ratio','negative_ratio','total_score','avg_num_comments']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# YouTube numeric conversion
for col in ['video_view_count','likes','trailer_score','comment_count',
            'yt_avg_compound','yt_avg_positive','yt_avg_negative','yt_avg_neutral',
            'yt_positive_ratio','yt_negative_ratio','total_likes','weighted_compound']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f"Zero Revenue : {(df['Revenue']==0).sum()} rows")
print(f"Zero Budget  : {(df['Budget']==0).sum()} rows")
print(f"Reddit coverage          : {(df['post_count']>0).sum()} / {len(df)} movies have Reddit posts")
print(f"YouTube comment coverage : {(df['comment_count']>0).sum()} / {len(df)} movies have trailer comments")
print(f"YouTube trailer coverage : {df['trailer_score'].notna().sum()} / {len(df)} movies have a trailer score")

### 2.2 Genre Multi-Label Encoding\n> Fitted on the **full dataset** so both revenue and rating subsets share identical columns.

In [ ]:
genre_list = df['Genre'].fillna('').str.split(',').apply(
    lambda x: [g.strip() for g in x if g.strip()])

mlb = MultiLabelBinarizer()
genre_matrix = pd.DataFrame(
    mlb.fit_transform(genre_list),
    columns=[f'genre_{g.replace(" ","_")}' for g in mlb.classes_],
    index=df.index
)
df = pd.concat([df, genre_matrix], axis=1)
GENRE_COLS = list(genre_matrix.columns)

print(f"Genres detected ({len(mlb.classes_)}): {list(mlb.classes_)}")
print(f"Genre columns added: {GENRE_COLS}")

### 2.3 Numeric, Studio, Budget, Reddit & YouTube Features

In [ ]:
MAJOR_STUDIOS = [
    'Warner Bros. Pictures','Universal Pictures','Paramount Pictures',
    'Marvel Studios','Walt Disney Pictures','Columbia Pictures',
    '20th Century Studios','Lionsgate','Netflix','Amazon Studios',
    'Apple Original Films','Sony Pictures'
]

# Studio flag
df['is_major_studio'] = df['Production_Companies'].apply(
    lambda x: int(any(s in str(x) for s in MAJOR_STUDIOS)))

# Budget features
df['budget_known'] = (df['Budget'].fillna(0) > 0).astype(int)
df['Budget']       = df['Budget'].fillna(0)
df['log_budget']   = np.log1p(df['Budget'])   # log-transform to reduce skew

# Cast size (number of billed actors)
df['cast_size'] = df['Cast'].fillna('').apply(
    lambda x: len([c for c in x.split(',') if c.strip()]))

# Runtime bucket: 0=short(<90m) 1=standard(90-120m) 2=long(120-150m) 3=epic(>150m)
df['runtime_bucket'] = pd.cut(df['Runtime'],
    bins=[0,90,120,150,9999], labels=[0,1,2,3]).astype(float)

# Reddit: has_reddit flag + fill NaN with neutral / zero
df['has_reddit'] = (df['post_count'].fillna(0) > 0).astype(int)
REDDIT_FILL = {
    'post_count':0,'avg_compound':0.0,'avg_positive':0.0,
    'avg_negative':0.0,'avg_neutral':0.5,'positive_ratio':0.0,
    'negative_ratio':0.0,'total_score':0,'avg_num_comments':0.0
}
for col, fill in REDDIT_FILL.items():
    df[col] = df[col].fillna(fill)

# YouTube features: log-transform heavily skewed counts (views/likes range from
# hundreds to ~100M); flag + neutral-fill missing comment sentiment (26 movies
# have comments disabled / all post-release per the 'notes' column).
df['log_view_count'] = np.log1p(df['video_view_count'].fillna(0))
df['log_likes']       = np.log1p(df['likes'].fillna(0))
df['log_total_likes'] = np.log1p(df['total_likes'].fillna(0))
df['trailer_score']   = df['trailer_score'].fillna(df['trailer_score'].median())

df['has_yt_comments'] = (df['comment_count'].fillna(0) > 0).astype(int)
YT_FILL = {
    'comment_count': 0, 'yt_avg_compound': 0.0, 'yt_avg_positive': 0.0,
    'yt_avg_negative': 0.0, 'yt_avg_neutral': 0.5, 'yt_positive_ratio': 0.0,
    'yt_negative_ratio': 0.0, 'weighted_compound': 0.0,
}
for col, fill in YT_FILL.items():
    df[col] = df[col].fillna(fill)

print("✅ Feature engineering complete")
df[['log_budget','budget_known','is_major_studio','cast_size','runtime_bucket',
    'has_reddit','avg_compound','log_view_count','trailer_score',
    'has_yt_comments','yt_avg_compound']].describe().round(3)

### 2.4 Define Feature Sets & Target Splits

In [ ]:
NUMERIC_FEATS = ['log_budget','budget_known','Runtime','runtime_bucket',
                 'Vote_Count','is_major_studio','cast_size','Year']
REDDIT_FEATS  = ['has_reddit','post_count','avg_compound','avg_positive',
                 'avg_negative','avg_neutral','positive_ratio',
                 'negative_ratio','total_score','avg_num_comments']
YOUTUBE_FEATS = ['log_view_count','log_likes','log_total_likes','trailer_score',
                  'has_yt_comments','comment_count','yt_avg_compound','yt_avg_positive',
                  'yt_avg_negative','yt_avg_neutral','yt_positive_ratio',
                  'yt_negative_ratio','weighted_compound']
ALL_FEATS = NUMERIC_FEATS + GENRE_COLS + REDDIT_FEATS + YOUTUBE_FEATS

print(f"Total features : {len(ALL_FEATS)}")
print(f"  Numeric/Studio : {len(NUMERIC_FEATS)}")
print(f"  Genre dummies  : {len(GENRE_COLS)}")
print(f"  Reddit         : {len(REDDIT_FEATS)}")
print(f"  YouTube        : {len(YOUTUBE_FEATS)}")

# Revenue dataset: only rows with known revenue (> 0)
# Rating  dataset: all rows
df_rev = df[df['Revenue'] > 0].copy().reset_index(drop=True)
df_rat = df.copy().reset_index(drop=True)

print(f"\nRevenue model dataset : {len(df_rev)} rows")
print(f"Rating  model dataset : {len(df_rat)} rows")

### 2.5 Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.suptitle('Exploratory Data Analysis', fontsize=15, fontweight='bold')

# Revenue distribution
ax = axes[0,0]
ax.hist(np.log1p(df_rev['Revenue']), bins=28, color='#3498db', edgecolor='white', alpha=0.85)
ax.set_title('Revenue Distribution (log₁ₚ scale)')
ax.set_xlabel('log(Revenue + 1)')
ax.set_ylabel('Count')

# Rating distribution
ax = axes[0,1]
ax.hist(df_rat['Rating'], bins=20, color='#2ecc71', edgecolor='white', alpha=0.85)
ax.axvline(7.0, color='red', linestyle='--', linewidth=1.5, label='Critical threshold (7.0)')
ax.set_title('Rating Distribution')
ax.set_xlabel('Rating (1-10)')
ax.legend(fontsize=9)

# Budget vs Revenue scatter
ax = axes[0,2]
mask = df_rev['Budget'] > 0
ax.scatter(np.log1p(df_rev.loc[mask,'Budget']),
           np.log1p(df_rev.loc[mask,'Revenue']),
           alpha=0.5, color='#9b59b6', s=40, edgecolors='none')
ax.set_title('Budget vs Revenue (log scale)')
ax.set_xlabel('log(Budget + 1)')
ax.set_ylabel('log(Revenue + 1)')

# Trailer views vs Revenue (NEW — YouTube)
ax = axes[0,3]
ax.scatter(df_rev['log_view_count'], np.log1p(df_rev['Revenue']),
           alpha=0.5, color='#e67e22', s=40, edgecolors='none')
ax.set_title('Trailer Views vs Revenue (log scale)')
ax.set_xlabel('log(Trailer Views + 1)')
ax.set_ylabel('log(Revenue + 1)')

# Reddit sentiment
ax = axes[1,0]
sent = df_rev.loc[df_rev['has_reddit']==1, 'avg_compound']
ax.hist(sent, bins=20, color='#e74c3c', edgecolor='white', alpha=0.85)
ax.axvline(0, color='black', linestyle='--', linewidth=1)
ax.set_title(f'Reddit Sentiment (n={len(sent)} movies with posts)')
ax.set_xlabel('Avg Compound Score (-1 to +1)')

# YouTube comment sentiment (NEW)
ax = axes[1,1]
yt_sent = df_rev.loc[df_rev['has_yt_comments']==1, 'yt_avg_compound']
ax.hist(yt_sent, bins=20, color='#1abc9c', edgecolor='white', alpha=0.85)
ax.axvline(0, color='black', linestyle='--', linewidth=1)
ax.set_title(f'YouTube Comment Sentiment (n={len(yt_sent)})')
ax.set_xlabel('Avg Compound Score (-1 to +1)')

# Top genres
ax = axes[1,2]
genre_sums = df_rev[GENRE_COLS].sum().sort_values(ascending=False).head(10)
labels = [g.replace('genre_','').replace('_',' ') for g in genre_sums.index]
ax.barh(labels[::-1], genre_sums.values[::-1], color='#f39c12', edgecolor='white')
ax.set_title('Top 10 Genres (revenue dataset)')
ax.set_xlabel('Count')

# Studio type vs revenue
ax = axes[1,3]
rev_major = df_rev.loc[df_rev['is_major_studio']==1, 'Revenue']
rev_indie  = df_rev.loc[df_rev['is_major_studio']==0, 'Revenue']
ax.boxplot([np.log1p(rev_indie), np.log1p(rev_major)],
           labels=['Indie / Other','Major Studio'],
           patch_artist=True,
           boxprops=dict(facecolor='#ecf0f1'),
           medianprops=dict(color='#e74c3c', linewidth=2))
ax.set_title('Revenue by Studio Type (log scale)')
ax.set_ylabel('log(Revenue + 1)')

for ax in axes.flatten():
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: eda_overview.png")

---
## 🤖 Phase 3 — Model Training

### 3.1 Prepare Feature Matrices & Train/Val/Test Split

In [ ]:
def build_xy(dataframe, target_col):
    X = dataframe[ALL_FEATS].copy()
    y = dataframe[target_col].copy()
    imputer = SimpleImputer(strategy='median')
    X_imp   = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
    return X_imp, y, imputer

# Revenue — log-transform target for better model fit
X_rev, y_rev_raw, imp_rev = build_xy(df_rev, 'Revenue')
y_rev = np.log1p(y_rev_raw)   # model learns in log space; we back-transform for metrics

# Rating — predict on original 1-10 scale
X_rat, y_rat, imp_rat = build_xy(df_rat, 'Rating')

def split_data(X, y, test_size=0.20, val_size=0.15, seed=42):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=test_size,
                                                random_state=seed)
    X_tr, X_va, y_tr, y_va = train_test_split(X_tr, y_tr,
                                                test_size=val_size/(1-test_size),
                                                random_state=seed)
    return X_tr, X_va, X_te, y_tr, y_va, y_te

X_tr_rev, X_va_rev, X_te_rev, y_tr_rev, y_va_rev, y_te_rev = split_data(X_rev, y_rev)
X_tr_rat, X_va_rat, X_te_rat, y_tr_rat, y_va_rat, y_te_rat = split_data(X_rat, y_rat)

print("Revenue split — Train:", len(X_tr_rev), "| Val:", len(X_va_rev), "| Test:", len(X_te_rev))
print("Rating  split — Train:", len(X_tr_rat), "| Val:", len(X_va_rat), "| Test:", len(X_te_rat))

### 3.2 Train Random Forest & Gradient Boosting (4 Models Total)

In [ ]:
# ── Random Forest ─────────────────────────────────────────────────────────────
rf_rev = RandomForestRegressor(n_estimators=300, max_depth=8,
                                min_samples_leaf=3, max_features='sqrt',
                                random_state=42, n_jobs=-1)
rf_rat = RandomForestRegressor(n_estimators=300, max_depth=6,
                                min_samples_leaf=3, max_features='sqrt',
                                random_state=42, n_jobs=-1)

# ── Gradient Boosting (substitute for XGBoost — see note in Cell 1) ──────────
xgb_rev = GradientBoostingRegressor(n_estimators=400, max_depth=5, learning_rate=0.03,
                                     subsample=0.8, random_state=42)
xgb_rat = GradientBoostingRegressor(n_estimators=400, max_depth=4, learning_rate=0.03,
                                     subsample=0.8, random_state=42)

rf_rev.fit(X_tr_rev,  y_tr_rev)
rf_rat.fit(X_tr_rat,  y_tr_rat)
xgb_rev.fit(X_tr_rev, y_tr_rev)
xgb_rat.fit(X_tr_rat, y_tr_rat)

print("✅ All 4 models trained successfully")

---
## 📊 Phase 4 — Evaluation

> Revenue metrics are reported in **original dollar scale** (back-transformed from log). Rating metrics are on the **1-10 scale**.

### 4.1 MAE · RMSE · R² on Test Set

In [ ]:
def evaluate_model(model, X_test, y_test, label, is_log=False):
    y_pred_t = model.predict(X_test)

    if is_log:
        y_pred = np.expm1(y_pred_t)
        y_true = np.expm1(y_test)
    else:
        y_pred = y_pred_t
        y_true = np.array(y_test)

    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    r2_t = r2_score(y_test, y_pred_t)   # log-space R² (revenue only)

    return {
        'label': label, 'is_log': is_log,
        'MAE': mae, 'RMSE': rmse, 'R2': r2, 'R2_log': r2_t,
        'y_true': y_true, 'y_pred': y_pred,
    }

res = {
    'rf_rev':  evaluate_model(rf_rev,  X_te_rev, y_te_rev, 'RF – Revenue',  is_log=True),
    'xgb_rev': evaluate_model(xgb_rev, X_te_rev, y_te_rev, 'GBR – Revenue', is_log=True),
    'rf_rat':  evaluate_model(rf_rat,  X_te_rat, y_te_rat, 'RF – Rating',   is_log=False),
    'xgb_rat': evaluate_model(xgb_rat, X_te_rat, y_te_rat, 'GBR – Rating',  is_log=False),
}

# ── Summary table ─────────────────────────────────────────────────────────────
rows = []
for k, r in res.items():
    if r['is_log']:
        rows.append({'Model': r['label'],
                     'MAE':  f"${r['MAE']:>18,.0f}",
                     'RMSE': f"${r['RMSE']:>18,.0f}",
                     'R²':    f"{r['R2']:.4f}",
                     'R² (log-space)': f"{r['R2_log']:.4f}"})
    else:
        rows.append({'Model': r['label'],
                     'MAE':  f"{r['MAE']:.4f}",
                     'RMSE': f"{r['RMSE']:.4f}",
                     'R²':    f"{r['R2']:.4f}",
                     'R² (log-space)': '—'})

metrics_df = pd.DataFrame(rows).set_index('Model')
print(metrics_df.to_string())
metrics_df

### 4.2 Actual vs Predicted Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle('Actual vs Predicted — Test Set', fontsize=14, fontweight='bold')

plot_order = [
    (axes[0,0], res['rf_rev']),
    (axes[0,1], res['xgb_rev']),
    (axes[1,0], res['rf_rat']),
    (axes[1,1], res['xgb_rat']),
]

for ax, r in plot_order:
    yt, yp = r['y_true'], r['y_pred']
    ax.scatter(yt, yp, alpha=0.55, s=45, color='#3498db', edgecolors='none')

    lo, hi = min(yt.min(), yp.min()), max(yt.max(), yp.max())
    ax.plot([lo, hi], [lo, hi], 'r--', linewidth=1.5, label='Perfect fit')

    ax.set_title(r['label'], fontsize=12, fontweight='bold')
    ax.set_xlabel('Actual')
    ax.set_ylabel('Predicted')

    if r['is_log']:
        stats = (f"MAE  = ${r['MAE']/1e6:.1f}M\n"
                 f"RMSE = ${r['RMSE']/1e6:.1f}M\n"
                 f"R²   = {r['R2']:.3f}")
        fmt = plt.FuncFormatter(lambda x,_: f'${x/1e6:.0f}M')
        ax.xaxis.set_major_formatter(fmt)
        ax.yaxis.set_major_formatter(fmt)
    else:
        stats = (f"MAE  = {r['MAE']:.3f}\n"
                 f"RMSE = {r['RMSE']:.3f}\n"
                 f"R²   = {r['R2']:.3f}")

    ax.text(0.04, 0.95, stats, transform=ax.transAxes, fontsize=9,
            va='top', bbox=dict(boxstyle='round,pad=0.4', fc='white',
                                ec='#bdc3c7', alpha=0.9))
    ax.legend(fontsize=9)
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: actual_vs_predicted.png")

### 4.3 Residual Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Residual Plots (Actual − Predicted)', fontsize=14, fontweight='bold')

for ax, r in plot_order:
    residuals = r['y_true'] - r['y_pred']
    ax.scatter(r['y_pred'], residuals, alpha=0.5, s=40,
               color='#9b59b6', edgecolors='none')
    ax.axhline(0, color='red', linestyle='--', linewidth=1.5)
    ax.set_title(r['label'], fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Residual')
    ax.spines[['top','right']].set_visible(False)
    if r['is_log']:
        fmt = plt.FuncFormatter(lambda x,_: f'${x/1e6:.0f}M')
        ax.xaxis.set_major_formatter(fmt)
        ax.yaxis.set_major_formatter(fmt)

plt.tight_layout()
plt.savefig('residual_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: residual_plots.png")

### 4.4 Feature Importance

In [ ]:
def feat_imp_plot(model, title, ax, top_n=15):
    imp = pd.Series(model.feature_importances_, index=ALL_FEATS)
    top = imp.sort_values(ascending=False).head(top_n)
    colors = ['#e74c3c' if f in REDDIT_FEATS else
              '#1abc9c' if f in YOUTUBE_FEATS else
              '#3498db' if f.startswith('genre_') else
              '#2ecc71' for f in top.index]
    top[::-1].plot(kind='barh', ax=ax, color=colors[::-1], edgecolor='white')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Importance')
    ax.spines[['top','right']].set_visible(False)

fig, axes = plt.subplots(2, 2, figsize=(16, 13))
fig.suptitle('Feature Importance — Top 15 per Model', fontsize=14, fontweight='bold')

feat_imp_plot(rf_rev,  'Random Forest — Revenue',  axes[0,0])
feat_imp_plot(xgb_rev, 'Gradient Boosting — Revenue', axes[0,1])
feat_imp_plot(rf_rat,  'Random Forest — Rating',   axes[1,0])
feat_imp_plot(xgb_rat, 'Gradient Boosting — Rating', axes[1,1])

from matplotlib.patches import Patch
legend = [Patch(color='#2ecc71', label='Numeric / Budget / Studio'),
          Patch(color='#3498db', label='Genre'),
          Patch(color='#e74c3c', label='Reddit Sentiment'),
          Patch(color='#1abc9c', label='YouTube Trailer/Sentiment')]
fig.legend(handles=legend, loc='lower center', ncol=4,
           fontsize=10, bbox_to_anchor=(0.5, -0.01))

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: feature_importance.png")

### 4.5 Ablation Study — Impact of Social Sentiment Sources
> Compares four feature configurations: metadata-only baseline, +Reddit, +YouTube, and +Both — using a fixed Random Forest so the comparison isolates feature-set effects rather than model-tuning effects.

In [ ]:
BASE_FEATS     = NUMERIC_FEATS + GENRE_COLS
BASE_REDDIT    = BASE_FEATS + REDDIT_FEATS
BASE_YOUTUBE   = BASE_FEATS + YOUTUBE_FEATS
BASE_BOTH      = BASE_FEATS + REDDIT_FEATS + YOUTUBE_FEATS

def ablation_r2_mae(feat_list, X_full, y_full, target_is_log, max_depth):
    X_sub = X_full[feat_list]
    Xtr, Xte, ytr, yte = train_test_split(X_sub, y_full, test_size=0.20, random_state=42)
    model = RandomForestRegressor(n_estimators=300, max_depth=max_depth,
                                   min_samples_leaf=3, random_state=42)
    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    if target_is_log:
        r2  = r2_score(np.expm1(yte), np.expm1(pred))
        mae = mean_absolute_error(np.expm1(yte), np.expm1(pred))
    else:
        r2  = r2_score(yte, pred)
        mae = mean_absolute_error(yte, pred)
    return r2, mae

configs = [
    ('Base only',          BASE_FEATS),
    ('+ Reddit',           BASE_REDDIT),
    ('+ YouTube',          BASE_YOUTUBE),
    ('+ Reddit + YouTube', BASE_BOTH),
]

ablation_rev, ablation_rat = {}, {}
for name, feats in configs:
    r2_r, mae_r = ablation_r2_mae(feats, X_rev, y_rev, True, 8)
    r2_t, mae_t = ablation_r2_mae(feats, X_rat, y_rat, False, 6)
    ablation_rev[name] = (r2_r, mae_r)
    ablation_rat[name] = (r2_t, mae_t)

print("Revenue Model — Random Forest:")
for name, (r2, mae) in ablation_rev.items():
    print(f"  {name:22s} →  R²: {r2:.4f}   MAE: ${mae:,.0f}")
print()
print("Rating Model — Random Forest:")
for name, (r2, mae) in ablation_rat.items():
    print(f"  {name:22s} →  R²: {r2:.4f}   MAE: {mae:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Ablation Study — Impact of Social Sentiment Sources (Random Forest)',
             fontsize=12, fontweight='bold')

names  = list(ablation_rev.keys())
colors = ['#95a5a6', '#e74c3c', '#1abc9c', '#9b59b6']

for ax, title, source in [(axes[0], 'Revenue Model — R²', ablation_rev),
                            (axes[1], 'Rating Model  — R²', ablation_rat)]:
    vals = [source[n][0] for n in names]
    bars = ax.bar(names, vals, color=colors, width=0.55, edgecolor='white')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_ylim(0, max(vals)*1.3 if max(vals) > 0 else 1)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylabel('R²')
    ax.tick_params(axis='x', rotation=15)
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('ablation_reddit.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: ablation_reddit.png")

### 4.6 5-Fold Cross-Validation

In [ ]:
def cv_summary(model, X, y, label, is_log=False):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_validate(model, X, y, cv=kf,
                            scoring=['r2','neg_mean_absolute_error',
                                     'neg_root_mean_squared_error'])
    r2   =  scores['test_r2'].mean()
    mae  = -scores['test_neg_mean_absolute_error'].mean()
    rmse = -scores['test_neg_root_mean_squared_error'].mean()
    if is_log:
        print(f"  {label:25s}  R²={r2:.4f}  MAE≈${np.expm1(mae):>15,.0f}  RMSE≈${np.expm1(rmse):>15,.0f}")
    else:
        print(f"  {label:25s}  R²={r2:.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}")

print("=== 5-Fold CV — Revenue Model (log-space) ===")
cv_summary(rf_rev,  X_rev, y_rev, "Random Forest",  is_log=True)
cv_summary(xgb_rev, X_rev, y_rev, "Gradient Boosting", is_log=True)

print()
print("=== 5-Fold CV — Rating Model ===")
cv_summary(rf_rat,  X_rat, y_rat, "Random Forest",  is_log=False)
cv_summary(xgb_rat, X_rat, y_rat, "Gradient Boosting", is_log=False)

---
## 🏆 Two-Stage Success Classification

Raw regression outputs → discrete success label using rule-based thresholds:

| Label | Revenue Condition | Rating Condition |
|---|---|---|
| 🏆 **Blockbuster Hit** | Revenue ≥ 2.5× Budget | Rating ≥ 7.0 |
| 💰 **Commercial Success** | Revenue ≥ 2.5× Budget | Rating < 7.0 |
| 🎭 **Critical Darling** | Revenue < 2.5× Budget | Rating ≥ 7.0 |
| 📉 **Flop** | Revenue < 2.5× Budget | Rating < 7.0 |

> When budget is unknown (= 0), a revenue proxy of **$100M** is used as the financial success threshold.

In [ ]:
REVENUE_MULTIPLIER = 2.5
RATING_THRESHOLD   = 7.0

def classify_success(pred_rev, pred_rat, budget):
    if budget > 0:
        fin_success = pred_rev >= REVENUE_MULTIPLIER * budget
    else:
        fin_success = pred_rev >= 100_000_000
    crit_success = pred_rat >= RATING_THRESHOLD

    if fin_success and crit_success:     return '🏆 Blockbuster Hit'
    elif fin_success:                    return '💰 Commercial Success'
    elif crit_success:                   return '🎭 Critical Darling'
    else:                                return '📉 Flop'

# Use best R² model per target
best_rev = rf_rev  if res['rf_rev']['R2']  >= res['xgb_rev']['R2']  else xgb_rev
best_rat = rf_rat  if res['rf_rat']['R2']  >= res['xgb_rat']['R2']  else xgb_rat

# Predict over entire revenue dataset (shared feature set)
X_rat_rev = imp_rat.transform(df_rev[ALL_FEATS])   # align rating imputer to revenue rows
X_rat_rev = pd.DataFrame(X_rat_rev, columns=ALL_FEATS)

pred_rev_all = np.expm1(best_rev.predict(X_rev))
pred_rat_all = best_rat.predict(X_rat_rev)

df_rev['pred_revenue']  = pred_rev_all
df_rev['pred_rating']   = pred_rat_all
df_rev['success_label'] = [
    classify_success(r, g, b)
    for r, g, b in zip(pred_rev_all, pred_rat_all, df_rev['Budget'])
]

counts = df_rev['success_label'].value_counts()
print("Predicted success category distribution:")
print(counts.to_string())

In [ ]:
PALETTE = {
    '🏆 Blockbuster Hit':    '#f39c12',
    '💰 Commercial Success': '#3498db',
    '🎭 Critical Darling':   '#2ecc71',
    '📉 Flop':               '#e74c3c',
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Two-Stage Success Classification', fontsize=13, fontweight='bold')

# Bar chart
order  = ['🏆 Blockbuster Hit','💰 Commercial Success',
           '🎭 Critical Darling','📉 Flop']
cnt    = [counts.get(c, 0) for c in order]
bars   = axes[0].bar([o.split(' ',1)[1] for o in order], cnt,
                      color=[PALETTE[o] for o in order],
                      edgecolor='white', width=0.55)
for bar, c in zip(bars, cnt):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.4,
                 str(c), ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[0].set_title('Category Distribution')
axes[0].set_ylabel('Movie Count')
axes[0].spines[['top','right']].set_visible(False)

# Scatter: predicted revenue vs rating
for cat in order:
    grp = df_rev[df_rev['success_label'] == cat]
    axes[1].scatter(grp['pred_revenue']/1e6, grp['pred_rating'],
                    label=cat.split(' ',1)[1], color=PALETTE[cat],
                    s=60, alpha=0.75, edgecolors='none')
axes[1].axhline(RATING_THRESHOLD, color='gray', linestyle='--',
                linewidth=1.2, label=f'Rating ≥ {RATING_THRESHOLD}')
axes[1].set_title('Predicted Revenue vs Rating')
axes[1].set_xlabel('Predicted Revenue ($M)')
axes[1].set_ylabel('Predicted Rating')
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x:.0f}M'))
axes[1].legend(fontsize=9, bbox_to_anchor=(1.01,1), loc='upper left')
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('success_categories.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: success_categories.png")

### Sample Predictions

In [ ]:
sample = df_rev[['Title','Budget','Revenue','Rating',
                  'pred_revenue','pred_rating','success_label']].copy()
sample['Budget (M)']          = sample['Budget'].apply(lambda x: f'${x/1e6:.1f}M' if x>0 else 'Unknown')
sample['Actual Revenue (M)']  = sample['Revenue'].apply(lambda x: f'${x/1e6:.1f}M')
sample['Pred Revenue (M)']    = sample['pred_revenue'].apply(lambda x: f'${x/1e6:.1f}M')
sample['Actual Rating']       = sample['Rating'].round(2)
sample['Pred Rating']         = sample['pred_rating'].round(2)

cols = ['Title','Budget (M)','Actual Revenue (M)','Pred Revenue (M)',
        'Actual Rating','Pred Rating','success_label']
sample[cols].head(20)

---
## 💾 Save All Outputs

In [ ]:
# Predictions CSV
df_rev[['Title','Release_Date','Budget','Revenue','Rating',
        'pred_revenue','pred_rating','success_label']].to_csv(
    'movie_predictions.csv', index=False)
print("✅ Saved: movie_predictions.csv")

# Metrics JSON
summary = {
    'revenue_model': {
        'Random Forest': {'MAE': res['rf_rev']['MAE'],  'RMSE': res['rf_rev']['RMSE'],  'R2': res['rf_rev']['R2']},
        'Gradient Boosting': {'MAE': res['xgb_rev']['MAE'], 'RMSE': res['xgb_rev']['RMSE'], 'R2': res['xgb_rev']['R2']},
    },
    'rating_model': {
        'Random Forest': {'MAE': res['rf_rat']['MAE'],  'RMSE': res['rf_rat']['RMSE'],  'R2': res['rf_rat']['R2']},
        'Gradient Boosting': {'MAE': res['xgb_rat']['MAE'], 'RMSE': res['xgb_rat']['RMSE'], 'R2': res['xgb_rat']['R2']},
    },
    'ablation_revenue': {name: {'R2': r2, 'MAE': mae} for name, (r2, mae) in ablation_rev.items()},
    'ablation_rating':  {name: {'R2': r2, 'MAE': mae} for name, (r2, mae) in ablation_rat.items()},
    'success_label_distribution': df_rev['success_label'].value_counts().to_dict(),
    'thresholds': {'revenue_multiplier': REVENUE_MULTIPLIER, 'rating_threshold': RATING_THRESHOLD},
    'dataset': {'revenue_rows': len(df_rev), 'rating_rows': len(df_rat), 'features': len(ALL_FEATS),
                'reddit_coverage': int((df['has_reddit']==1).sum()),
                'youtube_comment_coverage': int((df['has_yt_comments']==1).sum())}
}
with open('results_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print("✅ Saved: results_summary.json")

print()
print("=" * 55)
print("  PHASES 2–4 COMPLETE — Files generated:")
for f in ['eda_overview.png','actual_vs_predicted.png','residual_plots.png',
          'feature_importance.png','ablation_reddit.png','success_categories.png',
          'movie_predictions.csv','results_summary.json']:
    print(f"    ✅ {f}")
print("=" * 55)